🎧 **Audio introduction**

<audio controls>
  <source src="https://raw.githubusercontent.com/fair-vid/video-multimodal-pipeline/main/assets/audio/merge_document_pages.wav" type="audio/wav">
</audio>


---

## 📄 Document-Level Aggregation of Page-Wise LLM Outputs

This notebook implements a **document-level aggregation step** in the agent-based admissions process digital twin described in the accompanying paper.

In earlier steps of the pipeline, document images are processed **page by page** using a vision-capable local large language model (e.g., Gemma 3).
This design choice improves robustness and memory efficiency, but it results in **fragmented semantic outputs**, where each page is interpreted independently.

### Purpose of this Notebook

The goal of this notebook is to **reconstruct coherent, document-level representations** by merging page-wise LLM outputs into a single consolidated artifact per document.

Specifically, this notebook:

* groups LLM outputs that belong to the same original document,
* preserves page order during reconstruction,
* concatenates page-level interpretations into a unified document-level text,
* prepares inputs for downstream agents that require holistic reasoning (e.g., credential validation, consistency checks, and application-level assessment).

### Relation to the Digital Twin Architecture

In the full system described in the paper, this aggregation step is performed automatically as part of a **parallel agent workflow**, immediately after page-level vision-language inference.
For demonstration and reproducibility in Google Colab, the aggregation is implemented here as a separate notebook, making the transformation explicit and inspectable.

This step is critical for transparency: it allows reviewers and administrators to trace how **page-local interpretations** contribute to **document-level conclusions**, supporting auditability and human–AI collaboration.

### Output

For each application, the notebook produces:

* one consolidated text file per document,
* stored alongside the original page-level LLM outputs,
* ready for reconciliation with OCR results and higher-level decision agents.

> **Note:**
> This aggregation does not modify or reinterpret model outputs. It only restructures them to reflect the original document boundaries.

---



In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

✅ Google Drive is already mounted


In [ ]:
import os

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"

subdirs = [d for d in os.listdir(drive_dir) if os.path.isdir(os.path.join(drive_dir, d))]
print(subdirs)

['0_0', '60811_46042', '29696_25487']


In [ ]:
import re
from collections import defaultdict
from pathlib import Path

def group_ocr_files(file_list):
    page_pattern = re.compile(r'\s\(page \d+\)\.txt$')
    documents = defaultdict(list)
    for filename in file_list:
        if page_pattern.search(filename):
            base_name = page_pattern.sub('', filename)
        else:
            base_name = filename.rsplit('.txt', 1)[0]
        documents[base_name].append(filename)
    return dict(documents)

def read_and_concatenate_pages(pages):
    concatenated_text = []
    for page_path in sorted(pages):
        try:
            with open(page_path, 'r', encoding='utf-8') as f:
                text = f.read()
                concatenated_text.append(text)
        except Exception as e:
            print(f"Error reading {page_path}: {e}")
    return '\n'.join(concatenated_text)

def save_concatenated_documents(grouped_docs, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for doc_id, pages in grouped_docs.items():
        doc_name = Path(doc_id).name
        concatenated_text = read_and_concatenate_pages(pages)
        output_file = output_dir / f"{doc_name}.txt"
        try:
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(concatenated_text)
            print(f"Saved: {output_file}")
        except Exception as e:
            print(f"Error saving {output_file}: {e}")

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"
prefix = "documents_image_llm_text_"

for applicant_dir in Path(drive_dir).iterdir():
    if applicant_dir.is_dir():
        # Find subdirectories matching the pattern
        for target_dir in applicant_dir.glob(f"{prefix}*"):
            if target_dir.is_dir():
                model_name = target_dir.name[len(prefix):]
                output_doc_path = applicant_dir / f"{target_dir.name}_doc"

                if output_doc_path.exists():
                    print(f"Skipping {applicant_dir.name}/{target_dir.name} - output exists")
                    continue

                print(f"Processing: {target_dir} (Model: {model_name})")
                files = [str(f) for f in target_dir.iterdir() if f.is_file()]

                if not files:
                    continue

                grouped_docs = group_ocr_files(files)
                save_concatenated_documents(grouped_docs, output_doc_path)

                print(f"Aggregated {len(files)} files into {output_doc_path.name}")
                print("=" * 50)

Processing: /content/drive/MyDrive/colab-output/dream_applicant_application/0_0/documents_image_llm_text_glm-ocr (Model: glm-ocr)
Skipping 60811_46042/documents_image_llm_text_glm-ocr - output exists
Processing: /content/drive/MyDrive/colab-output/dream_applicant_application/60811_46042/documents_image_llm_text_glm-ocr_doc (Model: glm-ocr_doc)
Saved: /content/drive/MyDrive/colab-output/dream_applicant_application/60811_46042/documents_image_llm_text_glm-ocr_doc_doc/46042_CV- Saif.pdf.txt
Saved: /content/drive/MyDrive/colab-output/dream_applicant_application/60811_46042/documents_image_llm_text_glm-ocr_doc_doc/46042_MOI - Saif.pdf.txt
Saved: /content/drive/MyDrive/colab-output/dream_applicant_application/60811_46042/documents_image_llm_text_glm-ocr_doc_doc/46042_12th Online Result - Saif.pdf.txt
Saved: /content/drive/MyDrive/colab-output/dream_applicant_application/60811_46042/documents_image_llm_text_glm-ocr_doc_doc/46042_10th certificate cum marksheet - Saif.pdf.txt
Saved: /content/dr


---

### ▶️ Next Step: Executive Document Interpretation

After page-wise outputs have been aggregated into **coherent, document-level representations**, the next stage in the pipeline focuses on **executive interpretation and admissions-oriented summarization**.

In the following notebook, each reconstructed document is processed as a **single semantic unit** by a local large language model.
The model produces a structured executive summary that captures:

* the inferred document type (e.g., diploma, transcript, identification),
* authoritative dates, institutions, and durations,
* policy-constrained grade extraction (where applicable),
* a concise assessment of **international admissions relevance**.

This step reflects how human admissions officers interact with documents in practice: not by reading page fragments, but by evaluating **holistic, standardized summaries** that support comparison, triage, and decision-making.

📎 **Open the Executive Document Summarization notebook:**
[https://colab.research.google.com/drive/1wdxxvESwaSx3S99JUdL9_6hCNnCeCSUC?usp=sharing](https://colab.research.google.com/drive/1wdxxvESwaSx3S99JUdL9_6hCNnCeCSUC?usp=sharing)

> **Architectural note:**
> In the full digital twin described in the paper, document aggregation and executive summarization operate as parallel downstream agents once page-level inference is complete.
> They are separated here solely to improve transparency, reproducibility, and reviewer inspectability.

---
